In [15]:
import chromadb

DATA_DIR = "../data"

# Conectando ao Docker do ChromaDB
print("Conectando ao ChromaDB no Docker (localhost:8000)...")
cliente_chroma = chromadb.HttpClient(host="localhost", port=8000)
print("✅ Conexão estabelecida!")

Conectando ao ChromaDB no Docker (localhost:8000)...
✅ Conexão estabelecida!


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import json
from langchain_core.documents import Document

print("A carregar os ficheiros da pasta data...\n")

loader = DirectoryLoader(
    DATA_DIR, 
    glob="**/*.*",
    exclude=["dataset_finetuning_postech.jsonl", "dataset_rag.json"], 
    loader_cls=TextLoader
)

raw_documents = loader.load()

file_names = [doc.metadata['source'] for doc in raw_documents]
print(f"✅ {len(raw_documents)} ficheiro(s) carregado(s):")
for name in file_names:
    print(f" - {name}")
print("\n")

headers_to_split_on = [
        ("#", "Título Principal"),
        ("##", "Secção"),
        ("###", "Sub-secção"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

md_chunks = []
for doc in raw_documents:
    chunks = markdown_splitter.split_text(doc.page_content)
    for chunk in chunks:
        chunk.metadata.update(doc.metadata) 
    md_chunks.extend(chunks)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
final_chunks = text_splitter.split_documents(md_chunks)

print(f"✅ O texto total foi dividido em {len(final_chunks)} fragmentos otimizados.")

A carregar os ficheiros da pasta data...

✅ 5 ficheiro(s) carregado(s):
 - ../data/Diretriz_Dor_Toracica_IAM.md
 - ../data/Protocolo_Cetoacidose_Diabetica.md
 - ../data/Protocolo_Choque_Anafilatico.md
 - ../data/Protocolo_Crise_Asma.md
 - ../data/Diretriz_Profilaxia_Tetano.md


✅ O texto total foi dividido em 35 fragmentos otimizados.


In [17]:
import chromadb
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

print("A conectar ao ChromaDB no Docker (localhost:8000)...")
chroma_client = chromadb.HttpClient(host="localhost", port=8000)
print("✅ Conexão estabelecida!\n")

print("A gerar embeddings com o modelo bge-m3 e a enviar para o Docker...\n")

embeddings_model = OllamaEmbeddings(model="bge-m3")
collection_name = "hospital_protocols"

try:
    chroma_client.delete_collection(collection_name)
    print(f"Coleção '{collection_name}' anterior apagada para evitar duplicatas.")
except Exception:
    pass 

vectorstore = Chroma.from_documents(
    documents=final_chunks, 
    embedding=embeddings_model,
    client=chroma_client,
    collection_name=collection_name
)

print(f"🚀 SUCESSO! Dados inseridos na coleção '{collection_name}' com sucesso.")

A conectar ao ChromaDB no Docker (localhost:8000)...
✅ Conexão estabelecida!

A gerar embeddings com o modelo bge-m3 e a enviar para o Docker...

Coleção 'hospital_protocols' anterior apagada para evitar duplicatas.
🚀 SUCESSO! Dados inseridos na coleção 'hospital_protocols' com sucesso.
